# Modèle « personne au sol »

Détecter un ouvrier à terre — malaise, chute, accident.

Cinq cellules, dans l'ordre. Rien à choisir.

**Avant de commencer :** *Exécution → Modifier le type d'exécution → **T4 GPU***.


## Pourquoi ce modèle remplace l'existant

La chute est aujourd'hui une classe du modèle `gloves_glasses`, entraînée sur
trop peu d'exemples. Résultat : **elle alerte sur quelqu'un d'assis ou penché**.
On la contient avec un seuil relevé à 0,80, ce qui est un pansement — le modèle
reste faux, on le rend seulement plus silencieux.

Le jeu retenu corrige la cause. Il distingue trois postures :

| Classe | Ce que c'est | Doit alerter ? |
|---|---|---|
| `up` | debout | non |
| `bending` | penché, accroupi | **non** |
| `down` | à terre | **oui** |

Ce sont les 1 647 exemples de `bending` qui font la différence : ils apprennent
explicitement au modèle qu'un ouvrier qui se baisse n'est pas un ouvrier à terre.

**Un jeu à éviter :** le plus populaire de Roboflow (4 497 images, très bien
noté) ne contient qu'une seule classe, `Fall-Detected`. N'ayant jamais vu
personne debout, un modèle entraîné dessus ne peut pas apprendre à se taire —
c'est exactement le défaut qu'on cherche à corriger.


## 1 · Installation


In [ ]:
!pip install -q ultralytics roboflow

import torch
from ultralytics import YOLO

print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "AUCUN — activez le T4 dans Exécution > Modifier le type d'exécution")


## 2 · Télécharger le jeu de données

9 912 images après augmentation. Le téléchargement pèse environ 680 Mo :
comptez quelques minutes.


In [ ]:
from roboflow import Roboflow

CLE = 'kQle1ihpBmsqoXROy27k'    # votre clé Roboflow

rf = Roboflow(api_key=CLE)
projet = rf.workspace('yolo-h6urw').project('falling-zvpqk')
jeu = projet.version(1).download('yolov8')

import yaml
conf = yaml.safe_load(open(f'{jeu.location}/data.yaml'))
print('Dossier :', jeu.location)
print('Classes :', conf['names'])   # attendu : bending, down, up


## 3 · Entraîner

40 à 70 minutes sur un T4 — ce jeu est deux fois plus gros que celui des
camions. Laissez l'onglet ouvert.

`imgsz=640` n'est pas négociable : c'est la taille attendue par le serveur
Ciment's Eye.


In [ ]:
modele = YOLO('yolov8n.pt')

modele.train(
    data=f'{jeu.location}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=12,
    name='chute',
    # Une personne à terre peut être vue sous n'importe quel angle, mais
    # jamais tête en bas : on augmente la lumière et l'inclinaison, pas
    # le retournement vertical — qui apprendrait au modèle des postures
    # qui n'existent pas.
    hsv_v=0.5,
    degrees=10,
    fliplr=0.5,
    flipud=0.0,
)
print('Terminé.')


## 4 · Lire les résultats

Deux chiffres comptent, et ils se lisent ensemble :

- le **rappel sur `down`** : la part des personnes à terre effectivement vues.
  Une personne manquée, c'est un secours qui n'arrive pas ;
- la **précision sur `down`** : quand le modèle alerte, a-t-il raison ?
  Une précision faible signifie qu'il confond encore penché et à terre.

C'est le second qui dira si le problème actuel est réglé.


In [ ]:
meilleur = 'runs/detect/chute/weights/best.pt'
modele = YOLO(meilleur)
m = modele.val(data=f'{jeu.location}/data.yaml', imgsz=640, verbose=False)

print(f"{'posture':<12}{'précision':>11}{'rappel':>9}{'mAP50':>9}")
print('-' * 41)
for i, nom in modele.names.items():
    p, r, ap50, _ = m.box.class_result(i)
    marque = '  ← alerte' if nom in ('down', 'fallen') else ''
    print(f'{nom:<12}{p:>11.3f}{r:>9.3f}{ap50:>9.3f}{marque}')

print()
print('Repère : au-dessus de 0,75 en rappel ET en précision sur « down »,')
print('le modèle est exploitable en production.')


## 5 · Récupérer le modèle


In [ ]:
from google.colab import files
import shutil

shutil.copy(meilleur, 'ciments_eye_fall_best.pt')
files.download('ciments_eye_fall_best.pt')


---

## Installer sur le serveur

```powershell
copy %USERPROFILE%\Downloads\ciments_eye_fall_best.pt models\
.\venv\Scripts\python.exe scripts\export_openvino.py
```

Puis, dans `config/config.yaml`, passez le modèle `fall` à **`enabled: true`**.

La bascule est automatique : dès que `fall` est actif, la classe
`Fall-Detected` de `gloves_glasses` **se tait**. Sans cela, une même personne
au sol déclencherait deux alertes, et le décompte perdrait tout sens pour les
opérateurs.

La sévérité est déjà réglée sur **critique** : une personne à terre appelle un
secours immédiat, et le rappel se fait toutes les minutes tant que l'alerte
n'est pas prise en charge.
